# GH05T3 Fine-Tune — 2x T4
Standard transformers + bitsandbytes + PEFT (no unsloth — T4 CUDA arch incompatible)

**Kaggle settings:** Accelerator = GPU T4 x2 · Internet = On · Persistence = Files only

In [ ]:
# Cell 1: Install deps
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'transformers', 'peft', 'trl>=0.9.0', 'bitsandbytes', 'accelerate', 'datasets'], check=True)
print('deps ready')

In [ ]:
# Cell 2: GPU check
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB | sm_{p.major}{p.minor}')

In [ ]:
# Cell 3: Load data
# Tries /kaggle/input/gh05t3-datasets first (mounted dataset).
# If not mounted, downloads automatically via Kaggle CLI (pre-installed + auto-authed in Kaggle notebooks).
import json, random, subprocess
from pathlib import Path
from datasets import Dataset

DATA_DIR = Path('/kaggle/input/gh05t3-datasets')
if not DATA_DIR.exists():
    print('Dataset not mounted — downloading via Kaggle CLI...')
    dl_dir = Path('/tmp/gh05t3-data')
    dl_dir.mkdir(exist_ok=True)
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'tatortot/gh05t3-datasets',
         '--path', str(dl_dir), '--unzip'],
        check=True
    )
    DATA_DIR = dl_dir

print(f'Data dir: {DATA_DIR}')
print('Files:', sorted(p.name for p in DATA_DIR.iterdir()))

SYSTEM = (
    'You are GH05T3, an autonomous security and reasoning agent. '
    'You think carefully, reason step-by-step, and always prioritize '
    'detection and defense over exploitation.'
)

def read_jsonl(p):
    if not p.exists():
        print(f'  missing: {p}')
        return
    with open(p) as f:
        for line in f:
            line = line.strip()
            if line:
                try: yield json.loads(line)
                except: pass

def chatml(msgs):
    return '\n'.join(f'<|im_start|>{m["role"]}\n{m["content"]}<|im_end|>' for m in msgs)

texts = []

for rec in read_jsonl(DATA_DIR / 'adversarial_defense.jsonl'):
    t = rec.get('threat_vector', '')
    if not t: continue
    texts.append(chatml([
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': f'Analyze this threat:\n\n{t}'},
        {'role':'assistant', 'content':
            f"**Exploitation:** {rec.get('exploitation_method','N/A')}\n\n"
            f"**Detection:** {rec.get('detection_pattern','N/A')}\n\n"
            f"**Mitigation:** {rec.get('mitigation_strategy','N/A')}"},
    ]))

for rec in read_jsonl(DATA_DIR / 'reasoning_chains.jsonl'):
    q = rec.get('question', '')
    s = rec.get('reasoning_steps', [])
    if not q or not isinstance(s, list): continue
    texts.append(chatml([
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': q},
        {'role':'assistant', 'content':
            '**Reasoning:**\n' + '\n'.join(f'{i+1}. {x}' for i,x in enumerate(s)) +
            f"\n\n**Answer:** {rec.get('final_answer','N/A')}"},
    ]))

for rec in read_jsonl(DATA_DIR / 'cve_patterns.jsonl'):
    p = rec.get('vulnerability_pattern', '')
    if not p: continue
    ind = rec.get('discovery_indicators', [])
    texts.append(chatml([
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': f"Analyze {rec.get('source_cve','CVE')} vulnerability."},
        {'role':'assistant', 'content':
            f"**Pattern:** {p}\n\n"
            f"**Indicators:**\n" + ('\n'.join(f'• {x}' for x in ind) if isinstance(ind,list) else str(ind)) +
            f"\n\n**Lessons:** {rec.get('defensive_lessons','N/A')}"},
    ]))

for rec in read_jsonl(DATA_DIR / 'bug_bounty.jsonl'):
    tgt = rec.get('target_system', '')
    vuln = rec.get('vulnerability_found', '')
    if not tgt or not vuln: continue
    texts.append(chatml([
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': f'Security research on {tgt}: {vuln}'},
        {'role':'assistant', 'content':
            f"**Recon:** {rec.get('recon_method','N/A')}\n\n"
            f"**PoC:** {rec.get('non_weaponized_poc','N/A')}\n\n"
            f"**Remediation:** {rec.get('remediation','N/A')}"},
    ]))

assert texts, f'No data found at {DATA_DIR} — check dataset uploaded correctly'
random.seed(42)
random.shuffle(texts)
dataset = Dataset.from_dict({'text': texts})
print(f'Dataset: {len(dataset)} examples')
print('Sample:\n', dataset[0]['text'][:300], '...')

In [ ]:
# Cell 4: Load model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Give each GPU 13 GB headroom; hard-block CPU offload (bnb 4-bit doesn't support it)
n_gpus = torch.cuda.device_count()
max_memory = {i: '13GiB' for i in range(n_gpus)}
max_memory['cpu'] = '0GiB'
print(f'max_memory config: {max_memory}')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    max_memory=max_memory,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print('Model loaded')
if hasattr(model, 'hf_device_map'):
    from collections import Counter
    print('Device map:', dict(Counter(str(d) for d in model.hf_device_map.values())))
for i in range(n_gpus):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {used:.1f}/{total:.1f} GB used')

In [ ]:
# Cell 5: Apply LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

LORA_RANK = 32

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 6: Train
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

MAX_SEQ = 512

training_args = TrainingArguments(
    output_dir='outputs',
    max_steps=800,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    seed=42,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print('Training...')
stats = trainer.train()
print(f'Done — loss: {stats.training_loss:.4f}, steps: {stats.global_step}')

In [ ]:
# Cell 7: Save adapter
import json
OUT = 'gh05t3_lora_adapter'
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
with open(f'{OUT}/training_config.json', 'w') as f:
    json.dump({
        'model': MODEL_ID, 'lora_rank': LORA_RANK,
        'steps': stats.global_step, 'final_loss': stats.training_loss,
        'dataset_size': len(dataset),
    }, f, indent=2)
print(f'Adapter saved to /kaggle/working/{OUT}')

In [ ]:
# Cell 8: Quick inference test
from peft import PeftModel

model.eval()
prompt = '<|im_start|>system\nYou are GH05T3.<|im_end|>\n<|im_start|>user\nExplain SQL injection detection.<|im_end|>\n<|im_start|>assistant\n'
inputs = tokenizer(prompt, return_tensors='pt').to('cuda:0')
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))